# 01 — Data Audit


In [ ]:
%reload_ext autoreload
%autoreload 2
import os
import sys

sys.path.append('..')
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from tqdm import tqdm

from src.dataset import (
    average_hash,
    build_processed,
    check_no_leakage,
    get_embedding,
    make_imagefolders,
    make_train_val_split,
    md5_hash,
    scan_all_images,
)
from src.model import CNNBaseline
from src.utils import get_device, set_seed

DATA_ROOT='../data/raw/dataset'
os.listdir(DATA_ROOT)





In [ ]:
os.listdir(os.path.join(DATA_ROOT , 'train'))

In [ ]:

os.listdir(os.path.join(DATA_ROOT , 'unclean'))

In [ ]:
def count_classes(split_path):
    counts= {}
    for class_name in sorted(os.listdir(split_path)):
        class_dir = os.path.join (split_path , class_name)
        if os.path.isdir(class_dir):
            counts[class_name] = len(os.listdir(class_dir))
    return counts

train_count = count_classes (os.path.join(DATA_ROOT , 'train'))
print(train_count)

In [ ]:
def count_classes(split_path):
    counts= {}
    for class_name in sorted(os.listdir(split_path)):
        class_dir = os.path.join (split_path , class_name)
        if os.path.isdir(class_dir):
            counts[class_name] = len(os.listdir(class_dir))
    return counts

unclean_count = count_classes (os.path.join(DATA_ROOT , 'unclean'))
print(unclean_count)

In [ ]:
train_classes = set (train_count.keys())
unclean_classes = set (unclean_count.keys())

print("train:" , train_classes - unclean_classes)
print("unclean:" , unclean_classes - train_classes)
print("both:" , train_classes & unclean_classes)


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(train_count.keys(), train_count.values())
plt.xticks(rotation=45)
plt.title('Class distribution - train')
plt.ylabel('Number of samples')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(unclean_count.keys(), unclean_count.values())
plt.xticks(rotation=45)
plt.title('Class distribution - train')
plt.ylabel('Number of samples')
plt.tight_layout()
plt.show()

## MD5 & Average Hashing:

In [ ]:
records = scan_all_images(DATA_ROOT)
df = pd.DataFrame(records)
print(f"All scanned photos:{len(df)}")

md5_dupes = df [df.duplicated(subset='md5' , keep=False)]. sort_values('md5')
print(f"Duplicate groups with md5 : {md5_dupes['md5'].nunique()}")

avghash_dupes=df[df.duplicated(subset='avghash' , keep=False)].sort_values('avghash')
print(f"Duplicate groups with avghash: {avghash_dupes['avghash'].nunique()}")

def check_label_conflicts(dupes_df , hash_col):
    conflict = []
    for h , group in dupes_df.groupby(hash_col):
        if group['class'].nunique()>1:
            conflict.append(group)
    return conflict

md5_conflicts = check_label_conflicts(md5_dupes , 'md5')
avghash_conflicts=check_label_conflicts(avghash_dupes , 'avghash')
print(f"\nGroups with label conflicts(md5) :{len(md5_conflicts)}")
print(f"\nGroups with label conflicts(avghash): {len(avghash_conflicts)}")


train_df = df[df['split'] == 'train'].copy()
print(f"length of train img {len(train_df)}")



In [ ]:
def rescan(data_root=DATA_ROOT):
    global df
    df = scan_all_images(data_root)
    print(f"🔄 Rescanned: {len(df)} images")
    return df


In [ ]:
counts_by_split_class = df.groupby(['split', 'class']).size().reset_index(name='count')
print(counts_by_split_class.to_string())

print("\n--- جمع کل هر split ---")
print(df.groupby('split').size())

In [ ]:
def show_conflict_group(conflict_groups , group_index=0):
    group = conflict_groups[group_index]
    n = len(group)
    fig , axes = plt.subplots(1 , n , figsize=(4*n , 4))
    if n == 1:
        axes = [axes]
    for ax , (_, row) in zip (axes, group.iterrows()):
        with Image.open(row['path']) as img:
            ax.imshow(img)
        ax.set_title(f"{row['split']}/{row['class']}\n{row['filename']} , fontsize=9")
    plt.tight_layout()
    plt.show()

print("\nmd5 conflict group :")
if md5_conflicts:
    show_conflict_group(md5_conflicts, group_index=0)
else:
    print("No md5 conflicts found!")

for i in range(len(avghash_conflicts)):
    print(f"\n average hash conflicts{i}")
    print("_" * 20)
    show_conflict_group(avghash_conflicts , group_index=i)

## Resnet18:

In [ ]:
df = pd.DataFrame(scan_all_images(DATA_ROOT))
train_df = df[df['split'] == 'train'].copy().reset_index(drop=True)
print(f"train images: {len(train_df)}")


embeddings, valid_idx = [], []
for i, path in enumerate(tqdm(train_df['path'], desc="computing embeddings")):
    try:
        embeddings.append(get_embedding(path))
        valid_idx.append(i)
    except Exception as e:
        print(f"skipped {path}: {e}")

train_df = train_df.iloc[valid_idx].copy().reset_index(drop=True)
train_df['embedding'] = embeddings


train_df['centroid_distance'] = np.nan
for class_name, group in train_df.groupby('class'):
    class_embeddings = np.stack(group['embedding'].values)
    centroid = class_embeddings.mean(axis=0)
    train_df.loc[group.index, 'centroid_distance'] = np.linalg.norm(class_embeddings - centroid, axis=1)

suspects = (
    train_df.sort_values('centroid_distance', ascending=False)
    .groupby('class').head(15)
)



In [ ]:
def show_suspects(suspects_df, class_name, n=15):
    subset = suspects_df[suspects_df['class'] == class_name].head(n)
    cols = 5
    rows = (len(subset) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.5))
    for ax, (_, row) in zip(axes.flatten(), subset.iterrows()):
        with Image.open(row['path']) as img:
            ax.imshow(img)
        ax.set_title(f"d={row['centroid_distance']:.1f}\n{row['filename']}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_suspects(suspects, 'ambulance')

In [ ]:
def show_suspects(suspects_df, class_name, n=15):
    subset = suspects_df[suspects_df['class'] == class_name].head(n)
    cols = 5
    rows = (len(subset) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.5))
    for ax, (_, row) in zip(axes.flatten(), subset.iterrows()):
        with Image.open(row['path']) as img:
            ax.imshow(img)
        ax.set_title(f"d={row['centroid_distance']:.1f}\n{row['filename']}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_suspects(suspects, 'autobus')

In [ ]:
def show_suspects(suspects_df, class_name, n=15):
    subset = suspects_df[suspects_df['class'] == class_name].head(n)
    cols = 5
    rows = (len(subset) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.5))
    for ax, (_, row) in zip(axes.flatten(), subset.iterrows()):
        with Image.open(row['path']) as img:
            ax.imshow(img)
        ax.set_title(f"d={row['centroid_distance']:.1f}\n{row['filename']}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_suspects(suspects, 'kamyun')

In [ ]:
def show_suspects(suspects_df, class_name, n=15):
    subset = suspects_df[suspects_df['class'] == class_name].head(n)
    cols = 5
    rows = (len(subset) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.5))
    for ax, (_, row) in zip(axes.flatten(), subset.iterrows()):
        with Image.open(row['path']) as img:
            ax.imshow(img)
        ax.set_title(f"d={row['centroid_distance']:.1f}\n{row['filename']}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_suspects(suspects, 'kamyunet')

In [ ]:
def show_suspects(suspects_df, class_name, n=15):
    subset = suspects_df[suspects_df['class'] == class_name].head(n)
    cols = 5
    rows = (len(subset) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.5))
    for ax, (_, row) in zip(axes.flatten(), subset.iterrows()):
        with Image.open(row['path']) as img:
            ax.imshow(img)
        ax.set_title(f"d={row['centroid_distance']:.1f}\n{row['filename']}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_suspects(suspects, 'minibus')

In [ ]:
def show_suspects(suspects_df, class_name, n=15):
    subset = suspects_df[suspects_df['class'] == class_name].head(n)
    cols = 5
    rows = (len(subset) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.5))
    for ax, (_, row) in zip(axes.flatten(), subset.iterrows()):
        with Image.open(row['path']) as img:
            ax.imshow(img)
        ax.set_title(f"d={row['centroid_distance']:.1f}\n{row['filename']}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_suspects(suspects, 'savari')

In [ ]:
def show_suspects(suspects_df, class_name, n=15):
    subset = suspects_df[suspects_df['class'] == class_name].head(n)
    cols = 5
    rows = (len(subset) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.5))
    for ax, (_, row) in zip(axes.flatten(), subset.iterrows()):
        with Image.open(row['path']) as img:
            ax.imshow(img)
        ax.set_title(f"d={row['centroid_distance']:.1f}\n{row['filename']}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_suspects(suspects, 'taxi')

In [ ]:
def show_suspects(suspects_df, class_name, n=15):
    subset = suspects_df[suspects_df['class'] == class_name].head(n)
    cols = 5
    rows = (len(subset) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.5))
    for ax, (_, row) in zip(axes.flatten(), subset.iterrows()):
        with Image.open(row['path']) as img:
            ax.imshow(img)
        ax.set_title(f"d={row['centroid_distance']:.1f}\n{row['filename']}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_suspects(suspects, 'vanet')

## split:


In [ ]:
df = pd.DataFrame(scan_all_images(DATA_ROOT))
train_all=df[df['split'] == 'train'].reset_index(drop = True)
print(len(train_all))
print(train_all['class'].value_counts())

In [ ]:
print(pd.DataFrame({
    'train' :train_part['class'].value_counts(normalize = True),
    'val_part' : val_part['class'].value_counts(normalize=True),

}).round(3))


In [ ]:
SEED = 42
train_part, val_part = make_train_val_split(df, val_size=0.2, seed=SEED)
test_part = df[df['split'] == 'test'].reset_index(drop=True)

print(check_no_leakage(train_part, val_part, test_part))

##  Add size distribution and sample image grid:


In [ ]:
classes = sorted(train_all['class'].unique())
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for j, c in enumerate(classes):
    sample = train_all[train_all['class'] == c].sample(2, random_state=SEED)
    for i, (_, row) in enumerate(sample.iterrows()):
        with Image.open(row['path']) as img:
            axes[i, j].imshow(img)
        axes[i, j].set_title(c, fontsize=9)
        axes[i, j].axis('off')
plt.tight_layout()
plt.savefig('../results/sample_images.png', dpi=120)
plt.show()

In [ ]:
sizes = []
for p in df['path']:
    with Image.open(p) as img:
        sizes.append(img.size)
df['width'] = [s[0] for s in sizes]
df['height'] = [s[1] for s in sizes]
print(df[['width', 'height']].describe())
df[['width', 'height']].hist(bins=30)

## cnn baseline:

In [ ]:
based_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)) ,
    transforms.ToTensor() ,
    transforms.Normalize(mean=[0.485 , 0.456 , 0.406] , std= [ 0.229 , 0.224 , 0.225]),
])

train_ds = datasets.ImageFolder('../data/processed/train' , transform=based_transform)
val_ds = datasets.ImageFolder('../data/processed/val' , transform=based_transform)

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds , batch_size=32 , shuffle=True , generator= g , num_workers=0)
val_loader = DataLoader(val_ds , batch_size= 64 , shuffle= False , num_workers=0)

In [ ]:
x , y = next(iter(train_loader))
print(x.shape , y.shape)
model= CNNBaseline().to(device)
print(model(x.to(device)).shape)
print(sum(p.numel() for p in model.parameters()))